In [1]:
import pandas as pd 

In [2]:
text_db = pd.read_csv("text_db.csv")

In [3]:
cache_df = pd.read_json("retrieval_cache_19cc1c903ea1.json")

In [4]:
text_db.head(1)

,doc_id,date,document_class,metadata,wordcount,title,text
0,062cb58655d3ecfca9487a962b696864413a5970851344...,2025-04-13,news,"{'SourceCommonName': 'scoop.co.nz', 'SourceCol...",957.0,Worker Involvement Critical In Developing AI F...,A new PSA survey on the use of AI by public an...


In [5]:
cache_df.head(1)

,occupation,task,context_text,cached_docs,doc_titles,doc_sources,doc_chunk_ids
0,Chief Executives,Direct or coordinate an organization's financi...,SOURCE 1 [The exponential CFO]:\nTHE role of t...,[{'content': 'THE role of the chief financial ...,"[The exponential CFO, CFO vs CAO: Key Roles, D...","[news, news, news, news, news, news, news, news]",[6b58cd56acaf6c2f2aec7d7921254ffb6e4a671a1e3c2...


In [7]:
import json, csv, math, ast
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm

# ---------- Load article metadata (from text_db.csv) ----------
doc_meta = {}  # doc_id -> {"title": str, "date": datetime|None, "url": str|None}
with open("text_db.csv", newline="", encoding="utf-8") as f:
    r = csv.DictReader(f)
    for row in r:
        doc_id = row.get("doc_id")
        if not doc_id:
            continue

        d = None
        try:
            d = datetime.fromisoformat(row.get("date", ""))
        except Exception:
            pass

        url = None
        raw_meta = row.get("metadata") or ""
        if raw_meta:
            try:
                meta = ast.literal_eval(raw_meta)
                if isinstance(meta, dict):
                    url = (
                        meta.get("DocumentIdentifier")
                        or meta.get("url")
                        or meta.get("URL")
                        or meta.get("source_url")
                    )
            except Exception:
                pass

        # Keep newest duplicate row for the same doc_id
        cur = doc_meta.get(doc_id)
        if cur is None or (d and cur["date"] and d > cur["date"]) or (d and cur["date"] is None):
            doc_meta[doc_id] = {
                "title": row.get("title", ""),
                "date": d,
                "url": url,
            }
        elif cur.get("url") is None and url:
            # Fill missing URL even if we did not replace the row
            cur["url"] = url

# ---------- Load retrieval cache ----------
with open("retrieval_cache_19cc1c903ea1.json", encoding="utf-8") as f:
    cache = json.load(f)

# ---------- Build occupation -> docs stats ----------
occ_tasks = defaultdict(set)  # occ -> set(task)
occ_doc_tasks = defaultdict(lambda: defaultdict(set))  # occ -> doc_id -> set(task)
occ_doc_rankvals = defaultdict(lambda: defaultdict(list))  # occ -> doc_id -> list(rank_score)
occ_doc_title = defaultdict(dict)  # occ -> doc_id -> title
occ_doc_date = defaultdict(dict)  # occ -> doc_id -> datetime|None
occ_doc_url = defaultdict(dict)  # occ -> doc_id -> url|None

for row in tqdm(cache, desc="Parsing retrieval cache"):
    occ = row["occupation"]
    task = row["task"]
    occ_tasks[occ].add(task)

    chunk_ids = row.get("doc_chunk_ids", [])
    titles = row.get("doc_titles", [])

    for i, chunk_id in enumerate(chunk_ids):
        # chunk_id looks like "<doc_id>_chunk_<n>"
        doc_id = chunk_id.split("_chunk_")[0] if "_chunk_" in chunk_id else chunk_id
        meta = doc_meta.get(doc_id, {})
        title = titles[i] if i < len(titles) else meta.get("title", "")
        date = meta.get("date")
        url = meta.get("url")

        occ_doc_tasks[occ][doc_id].add(task)
        occ_doc_rankvals[occ][doc_id].append(1.0 / (1 + i))  # top ranks get higher value
        if doc_id not in occ_doc_title[occ]:
            occ_doc_title[occ][doc_id] = title
            occ_doc_date[occ][doc_id] = date
            occ_doc_url[occ][doc_id] = url

# ---------- Scoring helpers ----------
all_dates = [m["date"] for m in doc_meta.values() if m["date"] is not None]
global_max_date = max(all_dates) if all_dates else datetime.now()

def recency_score(d, now=global_max_date, half_life_days=180):
    """Exponential decay: 1 for newest, decays by half every ~180 days."""
    if d is None:
        return 0.35  # fallback for missing dates
    age_days = max((now - d).days, 0)
    return math.exp(-math.log(2) * age_days / half_life_days)

def choose_k(num_unique_docs, num_tasks, k_min=8, k_max=25):
    # dynamic k to keep UI manageable
    k = max(k_min, math.ceil(0.35 * num_tasks), math.ceil(math.sqrt(num_unique_docs)))
    return min(k, k_max)

# ---------- Rank + coverage-aware top-k ----------
occupation_top_articles = {}  # final output: occ -> list of article dicts

for occ in tqdm(occ_doc_tasks.keys(), desc="Reranking occupations"):
    total_tasks = len(occ_tasks[occ])
    doc_ids = list(occ_doc_tasks[occ].keys())

    # base score
    base = {}
    for doc_id in doc_ids:
        freq = len(occ_doc_tasks[occ][doc_id]) / total_tasks
        rank_q = sum(occ_doc_rankvals[occ][doc_id]) / len(occ_doc_rankvals[occ][doc_id])
        rec = recency_score(occ_doc_date[occ][doc_id])

        # weights (tuneable)
        score = 0.55 * freq + 0.25 * rec + 0.20 * rank_q
        base[doc_id] = {"score": score, "freq": freq, "recency": rec, "rank_q": rank_q}

    # greedy selection with task-coverage bonus
    k = choose_k(num_unique_docs=len(doc_ids), num_tasks=total_tasks)
    selected = []
    covered_tasks = set()
    remaining = set(doc_ids)

    for _ in range(min(k, len(doc_ids))):
        best_doc, best_score = None, -1
        for doc_id in remaining:
            new_tasks = occ_doc_tasks[occ][doc_id] - covered_tasks
            coverage_bonus = len(new_tasks) / total_tasks
            adjusted = base[doc_id]["score"] + 0.20 * coverage_bonus
            if adjusted > best_score:
                best_doc, best_score = doc_id, adjusted
        if best_doc is None:
            break
        selected.append(best_doc)
        covered_tasks |= occ_doc_tasks[occ][best_doc]
        remaining.remove(best_doc)

    articles = [
        {
            "doc_id": d,
            "title": occ_doc_title[occ].get(d, ""),
            "date": occ_doc_date[occ].get(d).date().isoformat() if occ_doc_date[occ].get(d) else None,
            "url": occ_doc_url[occ].get(d),
            "final_score": round(base[d]["score"], 4),
            "task_coverage": round(base[d]["freq"], 4),
            "recency": round(base[d]["recency"], 4),
            "rank_quality": round(base[d]["rank_q"], 4),
            "covered_tasks_count": len(occ_doc_tasks[occ][d]),
        }
        for d in selected
    ]

    # Keep highest final_score first for downstream display
    articles.sort(key=lambda x: x["final_score"], reverse=True)
    occupation_top_articles[occ] = articles

# save
with open("occupation_top_articles.json", "w", encoding="utf-8") as f:
    json.dump(occupation_top_articles, f, ensure_ascii=False, indent=2)

Reranking occupations: 100%|██████████| 923/923 [00:00<00:00, 6417.21it/s]


In [8]:
import json
import pandas as pd
from summa.summarizer import summarize
from tqdm.auto import tqdm

with open("occupation_top_articles_score_gt_0_5.json", "r", encoding="utf-8") as f:
    occ_to_articles = json.load(f)

# Flatten {occupation: [article, ...]} into a tabular dataframe
rows = [
    {"occupation": occ, **article}
    for occ, articles in occ_to_articles.items()
    for article in articles
]

df = pd.DataFrame(rows)

# Build occupation -> SOC mapping from ONET file
onet_map = (
    pd.read_csv("ONET_30.2_latest.csv", usecols=["Title", "O*NET-SOC Code"])
    .drop_duplicates()
    .rename(columns={"Title": "occupation", "O*NET-SOC Code": "soc_code"})
)

# Normalize to reduce mismatches caused by case/spacing
norm = lambda s: s.str.strip().str.lower()
df["occupation_norm"] = norm(df["occupation"])
onet_map["occupation_norm"] = norm(onet_map["occupation"])

# If one title maps to multiple SOC codes, keep the first code consistently
onet_map = onet_map.drop_duplicates(subset=["occupation_norm"], keep="first")

df = df.merge(onet_map[["occupation_norm", "soc_code"]], on="occupation_norm", how="left")
df = df.drop(columns=["occupation_norm"])

# Bring full article text from text_db using doc_id
text_lookup = (
    pd.read_csv("text_db.csv", usecols=["doc_id", "text"])
    .drop_duplicates(subset=["doc_id"], keep="first")
    .set_index("doc_id")["text"]
)
df["text"] = df["doc_id"].map(text_lookup)

def safe_summarize(txt, ratio=0.15):
    if not isinstance(txt, str):
        return ""
    txt = txt.strip()
    if len(txt.split()) < 30:
        return txt
    try:
        out = summarize(txt, ratio=ratio)
        return out if isinstance(out, str) and out.strip() else txt[:1200]
    except Exception:
        return txt[:1200]

tqdm.pandas(desc="Summarizing")
df["summary"] = df["text"].progress_apply(lambda x: safe_summarize(x, ratio=0.15))

df = df.sort_values(["occupation", "final_score"], ascending=[True, False]).reset_index(drop=True)


/Users/lucamouchel/anaconda3/envs/mit/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Summarizing: 100%|██████████| 3267/3267 [00:22<00:00, 142.53it/s]


In [9]:
df = df[["occupation", "soc_code", "doc_id", "date", "summary", "title", "url"]]

In [10]:
df

,occupation,soc_code,doc_id,date,summary,title,url
0,Accountants and Auditors,13-2011.00,a3cca3ae9403235a81cfff77467076a1c6543c3815e8ec...,2025-12-11,This explains why many organisations seek exte...,Understanding the Role of a Chartered Accounta...,https://www.chartsattack.com/chartered-account...
1,Accountants and Auditors,13-2011.00,83a75664ca761d5aa58069787f9f5d14d38609e2e0bc99...,2025-09-02,"FOR a long time, accountants were known mainly...",The accountant as business partner,https://www.manilatimes.net/2025/09/03/busines...
2,Accountants and Auditors,13-2011.00,d14ed16e84b2b9f5049c2a319f225069f6fcb50f30d0eb...,2025-07-23,Key responsibilities: Oversee the financial op...,Housing Authority of Grenada vacancy: Finance ...,https://nowgrenada.com/2025/07/housing-authori...
3,Actors,27-2011.00,dadb7c87eb98fd60a8bdc6add3b7f18c8e5d3e48b72948...,2025-11-14,The UCLA Department of Theater Summer Institut...,Step Into the Spotlight: Elevate Their Art wit...,https://www.broadwayworld.com/article/Step-Int...
4,Actors,27-2011.00,b0ab7162675cd6967403b4a8aecced56ddfcc411872aa0...,2026-02-16,It is difficult for education to keep pace wit...,How to train for an industry that won't wait –...,https://gadget.co.za/digitalindustry12m/
...,...,...,...,...,...,...,...
3262,Writers and Authors,27-3043.00,08a9b841ef0dd95a3eda3539789dbc8a237b52c5d9ecc5...,2026-02-26,Job Description: Piquant is on the lookout for...,The Irish Film & Television Network,https://www.iftn.ie/jobs/?act1=record&aid=18&r...
3263,Writers and Authors,27-3043.00,5cc118e69edd45a206319b790348e0e487cd1795408fdf...,2026-01-27,JournalismPakistan.com | Published: 27 January...,Career paths in journalism beyond reporting,https://www.journalismpakistan.com/career-path...
3264,Zoologists and Wildlife Biologists,19-1023.00,490883f75d86f623988431fc0939341e88e80401dc2917...,2025-11-05,"You'll dive into strategic planning, discover ...",Not Sure What to Study? Consider These 10 Majors,https://www.collegenews.com/article/consider-t...
3265,Zoologists and Wildlife Biologists,19-1023.00,7b924c23ca9b0a866469076169269dde0818c33206801f...,2026-02-23,"The VC, mentioned that the Institution has sev...","Embrace Digital Transformation, Entrepreneuria...",https://www.thetidenewsonline.com/2026/02/embr...


In [12]:
import re
import newspaper
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.request import Request, urlopen
from tqdm.auto import tqdm

# Normalize URL strings so mapping is consistent across the full df
df["url_norm"] = df["url"].astype("string").str.strip()
unique_urls = df["url_norm"].dropna().drop_duplicates().tolist()


def fetch_top_image(url: str):
   
    try:
        article = newspaper.article(url)
        img = getattr(article, "top_image", None)
        return img if isinstance(img, str) and img.strip() else None
    except Exception:
        return None


url_to_top_image = {}
max_workers = 16
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(fetch_top_image, u): u for u in unique_urls}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching top images (parallel)"):
        u = futures[fut]
        try:
            url_to_top_image[u] = fut.result()
        except Exception:
            url_to_top_image[u] = None

# Map back to every row in the full df by normalized URL key
missing_map_keys = int((~df["url_norm"].isna() & ~df["url_norm"].isin(url_to_top_image)).sum())
if missing_map_keys:
    print(f"Warning: {missing_map_keys} URLs were not present in url_to_top_image")

df["top_image"] = df["url_norm"].map(url_to_top_image)
df = df.drop(columns=["url_norm"])

# Save full dataframe with mapped top_image
df.to_parquet("df_occupation_news_with_images.parquet", index=False)
df.to_csv("df_occupation_news_with_images.csv", index=False)

print(f"Saved rows: {len(df)}")
print("Saved files: df_occupation_news_with_images.parquet, df_occupation_news_with_images.csv")

df[["url", "top_image"]].head()

Fetching top images (parallel):  45%|████▌     | 340/751 [00:31<00:36, 11.14it/s]/Users/lucamouchel/anaconda3/envs/mit/lib/python3.11/site-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname IST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
Fetching top images (parallel): 100%|██████████| 751/751 [01:39<00:00,  7.52it/s]

Saved rows: 3267
Saved files: df_occupation_news_with_images.parquet, df_occupation_news_with_images.csv


,url,top_image
0,https://www.chartsattack.com/chartered-account...,https://www.chartsattack.com/wp-content/upload...
1,https://www.manilatimes.net/2025/09/03/busines...,https://cdnx.premiumread.com/?url=https://www....
2,https://nowgrenada.com/2025/07/housing-authori...,https://nowgrenada.com/wp-content/uploads/2023...
3,https://www.broadwayworld.com/article/Step-Int...,https://cloudimages.broadwayworld.com/columnpi...
4,https://gadget.co.za/digitalindustry12m/,https://i0.wp.com/gadget.co.za/wp-content/uplo...
